In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

In [4]:
# Load and standardize Ejercicio.csv (column order can vary)
in_csv = Path("../data/Ejercicio.csv")
out_csv = Path("../data/geochem_ocr_raw.csv")  # keep downstream filename

assert in_csv.exists(), f"Input not found: {in_csv.resolve()}"

def _norm(col_name: str) -> str:
    return re.sub(r"[^a-z0-9]", "", str(col_name).lower())

# Canonical names expected by downstream calculations
aliases = {
    "Sample_ID": {"sampleid", "id", "muestra", "muestraid"},
    "Na": {"na", "sodio"},
    "K": {"k", "potasio"},
    "Ca": {"ca", "calcio"},
    "Mg": {"mg", "magnesio"},
    "Li": {"li", "litio"},
    "HCO3": {"hco3", "bicarbonato"},
    "Cl": {"cl", "ci", "chloride", "cloruro"},
    "SO4": {"so4", "sulfato", "sulphate"},
    "SiO2": {"sio2", "silice"},
    "B": {"b", "boro"},
    "F": {"f", "fluoruro", "fluoride"},
    "CO3": {"co3", "carbonato"},
    "PO4": {"po4", "fosfato", "phosphate"},
}

# Ejercicio.csv may use non-UTF8 encoding and sometimes has an extra pre-header row
best_df = None
best_score = -1
best_meta = None
for enc in ("utf-8", "latin-1", "cp1252"):
    for skip in (0, 1):
        try:
            df_try = pd.read_csv(in_csv, encoding=enc, skiprows=skip)
        except Exception:
            continue
        norm_cols = {_norm(c) for c in df_try.columns}
        score = sum(any(a in norm_cols for a in alias_set) for alias_set in aliases.values())
        if score > best_score:
            best_df = df_try
            best_score = score
            best_meta = (enc, skip)

if best_df is None:
    raise ValueError(f"Could not parse {in_csv} with tried encodings/skiprows.")

df_raw = best_df
print(f"Parsed with encoding={best_meta[0]}, skiprows={best_meta[1]}, alias_score={best_score}")

# Build rename map from present columns by normalized alias matching
norm_to_raw = {_norm(c): c for c in df_raw.columns}
rename_map = {}
for target, alias_set in aliases.items():
    for key in alias_set:
        if key in norm_to_raw:
            rename_map[norm_to_raw[key]] = target
            break

df_std = df_raw.rename(columns=rename_map)

# Keep only fields used by geochemical balance workflow
ion_cols = [
    "Na", "K", "Ca", "Mg", "Li", "Cl", "SO4", "HCO3", "CO3", "PO4", "B", "F", "SiO2",
]
available_ions = [c for c in ion_cols if c in df_std.columns]
if "Sample_ID" not in df_std.columns:
    raise ValueError("Could not find sample ID column (expected aliases like Muestra/ID/Sample_ID).")

df_table = df_std[["Sample_ID", *available_ions]].copy()

# Parse mixed numeric formats such as '<0.01', decimal comma, or blank values
def parse_numeric(series: pd.Series) -> pd.Series:
    s = series.astype(str).str.strip()
    s = s.str.replace(r"^<\s*", "", regex=True)
    s = s.str.replace(",", ".", regex=False)
    s = s.replace({"": np.nan, "-": np.nan, "nan": np.nan, "None": np.nan})
    return pd.to_numeric(s, errors="coerce")

# Units row in Ejercicio.csv becomes NaN and gets dropped
df_table["Sample_ID"] = parse_numeric(df_table["Sample_ID"])
for c in available_ions:
    df_table[c] = parse_numeric(df_table[c])

df_table = df_table.dropna(subset=["Sample_ID"]).copy()
df_table["Sample_ID"] = df_table["Sample_ID"].astype(int)
df_table = df_table.drop_duplicates().reset_index(drop=True)

# Save standardized dataset for downstream cells
df_table.to_csv(out_csv, index=False)
print(f"Saved: {out_csv.resolve()}")
print(f"Rows: {len(df_table)}, Cols: {len(df_table.columns)}")
print("Columns used:", list(df_table.columns))

df_table.head()

Parsed with encoding=latin-1, skiprows=1, alias_score=11
Saved: C:\Users\acb\geot\data\geochem_ocr_raw.csv
Rows: 27, Cols: 11
Columns used: ['Sample_ID', 'Na', 'K', 'Ca', 'Mg', 'Li', 'Cl', 'SO4', 'HCO3', 'B', 'SiO2']


,Sample_ID,Na,K,Ca,Mg,Li,Cl,SO4,HCO3,B,SiO2
0,1,19.4,6.8,30.0,5.0,0.0,6.0,2444.0,0.0,3.2,384.0
1,2,290.0,16.4,68.0,26.0,0.1,17.0,454.0,333.0,0.6,118.0
2,3,12.0,140.0,30.0,3.2,0.0,13.6,850.0,0.0,0.1,280.0
3,4,270.0,4.0,30.0,9.5,0.1,100.0,340.0,350.0,0.4,85.0
4,5,278.0,150.0,240.0,156.0,0.2,100.0,1439.0,344.0,0.8,161.0


In [5]:
# Quick validation of loaded table
required_base_cols = ["Sample_ID", "Na", "K", "Ca", "Mg", "Cl", "SO4", "HCO3"]
missing = [c for c in required_base_cols if c not in df_table.columns]
if missing:
    raise ValueError(f"Missing required columns for charge balance: {missing}")

print("Columns:", list(df_table.columns))
print("Null counts:")
print(df_table.isna().sum())

df_table.head(10)

Columns: ['Sample_ID', 'Na', 'K', 'Ca', 'Mg', 'Li', 'Cl', 'SO4', 'HCO3', 'B', 'SiO2']
Null counts:
Sample_ID    0
Na           0
K            0
Ca           0
Mg           0
Li           0
Cl           0
SO4          0
HCO3         0
B            0
SiO2         0
dtype: int64


,Sample_ID,Na,K,Ca,Mg,Li,Cl,SO4,HCO3,B,SiO2
0,1,19.4,6.8,30.0,5.0,0.00,6.0,2444.0,0.0,3.2,384.0
1,2,290.0,16.4,68.0,26.0,0.10,17.0,454.0,333.0,0.6,118.0
2,3,12.0,140.0,30.0,3.2,0.00,13.6,850.0,0.0,0.1,280.0
3,4,270.0,4.0,30.0,9.5,0.10,100.0,340.0,350.0,0.4,85.0
4,5,278.0,150.0,240.0,156.0,0.20,100.0,1439.0,344.0,0.8,161.0
5,6,55.9,5.9,54.0,4.8,0.01,76.8,23.0,230.0,0.0,81.6
6,7,81.5,4.9,36.4,18.8,0.01,42.3,14.8,273.0,0.2,59.0
7,8,46.0,3.6,54.4,35.6,0.01,42.3,11.6,283.0,0.7,79.0
8,9,67.0,3.4,53.8,28.6,0.01,92.2,21.2,253.0,1.3,83.0
9,10,513.0,9.3,223.8,31.6,0.22,192.0,1477.0,249.0,1.4,94.0


In [6]:
# Keep a clean working copy for all downstream calculations
# (already standardized in previous cell)
df_table = df_table.drop_duplicates().reset_index(drop=True)

# Re-save to ensure downstream cells use cleaned table
df_table.to_csv(out_csv, index=False)
print(f"Saved clean table: {out_csv.resolve()}")
print(f"Rows exported: {len(df_table)}")

df_table

Saved clean table: C:\Users\acb\geot\data\geochem_ocr_raw.csv
Rows exported: 27


,Sample_ID,Na,K,Ca,Mg,Li,Cl,SO4,HCO3,B,SiO2
0,1,19.4,6.8,30.0,5.00,0.00,6.0,2444.0,0.0,3.2,384.0
1,2,290.0,16.4,68.0,26.00,0.10,17.0,454.0,333.0,0.6,118.0
2,3,12.0,140.0,30.0,3.20,0.00,13.6,850.0,0.0,0.1,280.0
3,4,270.0,4.0,30.0,9.50,0.10,100.0,340.0,350.0,0.4,85.0
4,5,278.0,150.0,240.0,156.00,0.20,100.0,1439.0,344.0,0.8,161.0
5,6,55.9,5.9,54.0,4.80,0.01,76.8,23.0,230.0,0.0,81.6
6,7,81.5,4.9,36.4,18.80,0.01,42.3,14.8,273.0,0.2,59.0
7,8,46.0,3.6,54.4,35.60,0.01,42.3,11.6,283.0,0.7,79.0
8,9,67.0,3.4,53.8,28.60,0.01,92.2,21.2,253.0,1.3,83.0
9,10,513.0,9.3,223.8,31.60,0.22,192.0,1477.0,249.0,1.4,94.0


In [13]:
# Retrieve molecular weights (g/mol) and ion charge for species in geochem_ocr_raw.csv
mw_input_csv = Path("../data/geochem_ocr_raw.csv")
mw_output_csv = Path("../data/geochem_molecular_weights.csv")

assert mw_input_csv.exists(), f"Missing input file: {mw_input_csv.resolve()}"

df_geochem = pd.read_csv(mw_input_csv)
species_cols = [c for c in df_geochem.columns if c != "Sample_ID"]

# Standard atomic weights (g/mol)
atomic_weights = {
    "H": 1.008,
    "Li": 6.94,
    "B": 10.81,
    "C": 12.011,
    "O": 15.999,
    "F": 18.998403163,
    "Na": 22.98976928,
    "Mg": 24.305,
    "Si": 28.085,
    "S": 32.06,
    "Cl": 35.45,
    "K": 39.0983,
    "Ca": 40.078,
    "P": 30.973761998,
}

# Ionic charge (valence) used in charge-balance calculations
ion_charges = {
    "Na": 1,
    "K": 1,
    "Ca": 2,
    "Mg": 2,
    "Li": 1,
    "Cl": -1,
    "SO4": -2,
    "HCO3": -1,
    "CO3": -2,
    "PO4": -3,
    "B": 3,
    "F": -1,
    "SiO2": 0,
}

formula_token_re = re.compile(r"([A-Z][a-z]?)(\d*)")

def molar_mass(formula: str) -> float:
    formula = formula.replace("^", "")
    formula = re.sub(r"[+-]\d*$", "", formula)
    formula = formula.replace("+", "").replace("-", "")

    total = 0.0
    parsed = formula_token_re.findall(formula)
    if not parsed:
        raise ValueError(f"Could not parse formula: {formula}")

    for elem, count_str in parsed:
        if elem not in atomic_weights:
            raise KeyError(f"Atomic weight not found for element: {elem}")
        count = int(count_str) if count_str else 1
        total += atomic_weights[elem] * count
    return total

mw_rows = []
for sp in species_cols:
    mass = molar_mass(sp)
    mw_rows.append(
        {
            "species": sp,
            "molecular_weight_g_mol": round(mass, 6),
            "ion_charge": ion_charges.get(sp),
        }
    )

df_mw = pd.DataFrame(mw_rows)

# Compatibility guard: keep boron as trivalent cation across downstream cells
if "B" in set(df_mw["species"]):
    df_mw.loc[df_mw["species"] == "B", "ion_charge"] = 3

df_mw.to_csv(mw_output_csv, index=False)

print(f"Saved: {mw_output_csv.resolve()}")
df_mw

Saved: C:\Users\acb\geot\data\geochem_molecular_weights.csv


,species,molecular_weight_g_mol,ion_charge
0,Na,22.989769,1
1,K,39.098300,1
2,Ca,40.078000,2
3,Mg,24.305000,2
4,Li,6.940000,1
5,Cl,35.450000,-1
6,SO4,96.056000,-2
7,HCO3,61.016000,-1
8,B,10.810000,3
9,SiO2,60.083000,0


In [14]:
# Convert mg/dm3 to mmol/dm3 with species in rows and samples in columns
in_csv = Path("../data/geochem_ocr_raw.csv")
assert in_csv.exists(), f"Missing input file: {in_csv.resolve()}"

df_mg = pd.read_csv(in_csv)

# Build a mapping: species -> molecular weight (g/mol)
mw_map = dict(zip(df_mw["species"], df_mw["molecular_weight_g_mol"]))

species = [c for c in df_mg.columns if c != "Sample_ID"]

# Keep only species for which molecular weight is available
species = [s for s in species if s in mw_map]

# Convert to numeric matrix with samples as index
mg_matrix = df_mg.set_index("Sample_ID")[species].apply(pd.to_numeric, errors="coerce")

# mg/dm3 (mg/L) -> mmol/dm3 (mmol/L): mmol/L = mg/L / (g/mol)
mmol_matrix = mg_matrix.copy()
for sp in species:
    mmol_matrix[sp] = mg_matrix[sp] / mw_map[sp]

# Reorient so rows are species and columns are samples
df_mmol = mmol_matrix.T
df_mmol.index.name = "species"

# Optional export
mmol_out_csv = Path("../data/geochem_mmol_dm3.csv")
df_mmol.to_csv(mmol_out_csv)
print(f"Saved: {mmol_out_csv.resolve()}")

df_mmol

Saved: C:\Users\acb\geot\data\geochem_mmol_dm3.csv


Sample_ID,1,2,3,4,5,6,7,8,9,10,...,18,19,20,21,22,23,24,25,26,27
species,,,,,,,,,,,,,,,,,,,,,
Na,0.843854,12.614307,0.521971,11.744355,12.092336,2.431516,3.545055,2.000890,2.914340,22.314274,...,33.406164,17.094561,4.262766,6.089665,196.826684,2.000890,178.775176,231.972753,213.877747,145.456007
K,0.173921,0.419456,3.580718,0.102306,3.836484,0.150902,0.125325,0.092076,0.086960,0.237862,...,0.764739,0.432244,0.127883,0.260881,18.824348,0.230189,17.724556,23.607165,19.898563,9.667940
Ca,0.748540,1.696691,0.748540,0.748540,5.988323,1.347373,0.908229,1.357353,1.342382,5.584111,...,3.632916,3.937322,1.671740,1.023005,6.013274,1.397275,2.295524,4.915415,7.959479,7.560257
Mg,0.205719,1.069739,0.131660,0.390866,6.418432,0.197490,0.773503,1.464719,1.176713,1.300144,...,0.320922,0.053487,0.213948,0.238634,0.014812,0.020572,0.004114,0.004114,0.011109,0.011109
Li,0.000000,0.014409,0.000000,0.014409,0.028818,0.001441,0.001441,0.001441,0.001441,0.031700,...,0.027378,0.015850,0.004323,0.001441,3.688761,0.028818,3.429395,4.293948,4.005764,0.778098
Cl,0.169252,0.479549,0.383639,2.820874,2.820874,2.166432,1.193230,1.193230,2.600846,5.416079,...,18.110014,8.772920,3.244006,3.131171,227.108604,1.382228,206.911142,270.832158,241.297602,151.198872
SO4,25.443491,4.726410,8.849005,3.539602,14.980845,0.239444,0.154077,0.120763,0.220705,15.376447,...,8.495045,6.079787,0.233197,0.327934,0.801616,0.156159,0.718331,0.437245,1.030649,0.541351
HCO3,0.000000,5.457585,0.000000,5.736200,5.637865,3.769503,4.474236,4.638128,4.146453,4.080897,...,7.571784,5.572309,5.867313,5.490363,1.589747,2.294480,1.573358,1.114462,1.868362,0.016389
B,0.296022,0.055504,0.009251,0.037003,0.074006,0.000000,0.018501,0.064755,0.120259,0.129510,...,0.092507,0.046253,0.074006,0.055504,17.576318,0.555042,8.695652,17.206290,17.113784,4.162812


In [15]:
# Convert mmol/dm3 to meq/dm3 using ionic charge
# meq/dm3 = mmol/dm3 * ionic_charge
charge_map = dict(zip(df_mw["species"], df_mw["ion_charge"]))

# Compatibility guard for notebooks resumed from old state
if "B" in charge_map:
    charge_map["B"] = 3

# Align charges to species index in df_mmol
charge_series = pd.Series(charge_map).reindex(df_mmol.index)

# Signed meq (cations positive, anions negative)
df_meq = df_mmol.mul(charge_series, axis=0)

meq_out_csv = Path("../data/geochem_meq_dm3.csv")
df_meq.to_csv(meq_out_csv)
print(f"Saved: {meq_out_csv.resolve()}")

df_meq

Saved: C:\Users\acb\geot\data\geochem_meq_dm3.csv


Sample_ID,1,2,3,4,5,6,7,8,9,10,...,18,19,20,21,22,23,24,25,26,27
species,,,,,,,,,,,,,,,,,,,,,
Na,0.843854,12.614307,0.521971,11.744355,12.092336,2.431516,3.545055,2.000890,2.914340,22.314274,...,33.406164,17.094561,4.262766,6.089665,196.826684,2.000890,178.775176,231.972753,213.877747,145.456007
K,0.173921,0.419456,3.580718,0.102306,3.836484,0.150902,0.125325,0.092076,0.086960,0.237862,...,0.764739,0.432244,0.127883,0.260881,18.824348,0.230189,17.724556,23.607165,19.898563,9.667940
Ca,1.497081,3.393383,1.497081,1.497081,11.976646,2.694745,1.816458,2.714706,2.684765,11.168222,...,7.265832,7.874644,3.343480,2.046010,12.026548,2.794551,4.591047,9.830830,15.918958,15.120515
Mg,0.411438,2.139477,0.263320,0.781732,12.836865,0.394980,1.547007,2.929438,2.353425,2.600288,...,0.641843,0.106974,0.427895,0.477268,0.029624,0.041144,0.008229,0.008229,0.022218,0.022218
Li,0.000000,0.014409,0.000000,0.014409,0.028818,0.001441,0.001441,0.001441,0.001441,0.031700,...,0.027378,0.015850,0.004323,0.001441,3.688761,0.028818,3.429395,4.293948,4.005764,0.778098
Cl,-0.169252,-0.479549,-0.383639,-2.820874,-2.820874,-2.166432,-1.193230,-1.193230,-2.600846,-5.416079,...,-18.110014,-8.772920,-3.244006,-3.131171,-227.108604,-1.382228,-206.911142,-270.832158,-241.297602,-151.198872
SO4,-50.886983,-9.452819,-17.698009,-7.079204,-29.961689,-0.478887,-0.308154,-0.241526,-0.441409,-30.752894,...,-16.990089,-12.159574,-0.466395,-0.655867,-1.603231,-0.312318,-1.436662,-0.874490,-2.061298,-1.082702
HCO3,-0.000000,-5.457585,-0.000000,-5.736200,-5.637865,-3.769503,-4.474236,-4.638128,-4.146453,-4.080897,...,-7.571784,-5.572309,-5.867313,-5.490363,-1.589747,-2.294480,-1.573358,-1.114462,-1.868362,-0.016389
B,0.888067,0.166512,0.027752,0.111008,0.222017,0.000000,0.055504,0.194265,0.360777,0.388529,...,0.277521,0.138760,0.222017,0.166512,52.728955,1.665125,26.086957,51.618871,51.341351,12.488437


In [16]:
# Cation and anion sums (meq/dm3) per sample
charge_map = dict(zip(df_mw["species"], df_mw["ion_charge"]))

# Compatibility guard for notebooks resumed from old state
if "B" in charge_map:
    charge_map["B"] = 3

charges = pd.Series(charge_map).reindex(df_meq.index)

cation_species = charges[charges > 0].index
anion_species = charges[charges < 0].index

# Sums are computed per sample (column-wise)
df_cation_sum = pd.DataFrame([df_meq.loc[cation_species].sum(axis=0)], index=["cation_sum_meq_dm3"])
df_anion_sum = pd.DataFrame([df_meq.loc[anion_species].abs().sum(axis=0)], index=["anion_sum_meq_dm3"])

# Optional exports
cation_out_csv = Path("../data/geochem_cation_sum_meq_dm3.csv")
anion_out_csv = Path("../data/geochem_anion_sum_meq_dm3.csv")
df_cation_sum.to_csv(cation_out_csv)
df_anion_sum.to_csv(anion_out_csv)

print(f"Saved: {cation_out_csv.resolve()}")
print(f"Saved: {anion_out_csv.resolve()}")

df_cation_sum, df_anion_sum

Saved: C:\Users\acb\geot\data\geochem_cation_sum_meq_dm3.csv
Saved: C:\Users\acb\geot\data\geochem_anion_sum_meq_dm3.csv


(Sample_ID                1          2         3          4          5   \
 cation_sum_meq_dm3  3.81436  18.747544  5.890843  14.250891  40.993165   
 
 Sample_ID                 6        7         8         9          10  ...  \
 cation_sum_meq_dm3  5.673585  7.09079  7.932816  8.401708  36.740875  ...   
 
 Sample_ID                  18         19        20        21          22  \
 cation_sum_meq_dm3  42.383477  25.663033  8.388364  9.041778  284.124919   
 
 Sample_ID                 23          24          25        26          27  
 cation_sum_meq_dm3  6.760717  230.615359  321.331795  305.0646  183.533214  
 
 [1 rows x 27 columns],
 Sample_ID                 1          2          3          4          5   \
 anion_sum_meq_dm3  51.056235  15.389953  18.081648  15.636279  38.420429   
 
 Sample_ID                6        7         8         9         10  ...  \
 anion_sum_meq_dm3  6.414822  5.97562  6.072883  7.188709  40.24987  ...   
 
 Sample_ID                 18         19  

In [17]:
# Final dataframe: charge-balance error (%) per sample
# CBE(%) = ((sum_cations - sum_anions) / (sum_cations + sum_anions)) * 100

sum_c = df_cation_sum.loc["cation_sum_meq_dm3"]
sum_a = df_anion_sum.loc["anion_sum_meq_dm3"]

df_cbe = pd.DataFrame(
    [((sum_c - sum_a) / (sum_c + sum_a)) * 100],
    index=["cbe_percent"],
)

cbe_out_csv = Path("../data/geochem_charge_balance_error_percent.csv")
df_cbe.to_csv(cbe_out_csv)
print(f"Saved: {cbe_out_csv.resolve()}")

df_cbe

Saved: C:\Users\acb\geot\data\geochem_charge_balance_error_percent.csv


Sample_ID,1,2,3,4,5,6,7,8,9,10,...,18,19,20,21,22,23,24,25,26,27
cbe_percent,-86.09689,9.835495,-50.853312,-4.635392,3.239667,-6.131803,8.534635,13.279826,7.780416,-4.557684,...,-0.339086,-1.613578,-6.619976,-1.286211,10.462785,25.783782,4.697499,8.16468,10.873746,9.300879


In [18]:
# Flag samples that meet 5% charge-balance threshold
# pass if abs(CBE%) <= 5

df_cbe_flag = pd.DataFrame(index=df_cbe.columns)
df_cbe_flag.index.name = "Sample_ID"
df_cbe_flag["cbe_percent"] = df_cbe.loc["cbe_percent"].values
df_cbe_flag["abs_cbe_percent"] = df_cbe_flag["cbe_percent"].abs()
df_cbe_flag["passes_5pct_threshold"] = df_cbe_flag["abs_cbe_percent"] <= 5

flag_out_csv = Path("../data/geochem_charge_balance_flag_5pct.csv")
df_cbe_flag.to_csv(flag_out_csv)
print(f"Saved: {flag_out_csv.resolve()}")

df_cbe_flag

Saved: C:\Users\acb\geot\data\geochem_charge_balance_flag_5pct.csv


,cbe_percent,abs_cbe_percent,passes_5pct_threshold
Sample_ID,,,
1,-86.096890,86.096890,False
2,9.835495,9.835495,False
3,-50.853312,50.853312,False
4,-4.635392,4.635392,True
5,3.239667,3.239667,True
6,-6.131803,6.131803,False
7,8.534635,8.534635,False
8,13.279826,13.279826,False
9,7.780416,7.780416,False
